# Bronze Layer Data Ingestion - Retail Store Sales

## Purpose
This notebook ingests raw CSV sales data from Unity Catalog volumes into the **bronze layer** Delta table.

## Process Flow
1. **Setup**: Import schema types
2. **Read**: Load CSV from `/Volumes/sales/default/raw/retail_store_sales.csv` with explicit schema
3. **Standardize**: Rename columns from "Title Case" to "snake_case"
4. **Write**: Save as Delta table to `sales.bronze.retail_store_sales`

## Source Data
- File: `/Volumes/sales/default/raw/retail_store_sales.csv`

## Output
- Delta table: `sales.bronze.retail_store_sales` with standardized column names

In [0]:
# ============================================================================
# Setup: Import Libraries and Define Table Name
# ============================================================================

# Hardcoded table name
table = "retail_store_sales"

# Import PySpark schema types for explicit schema definition
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, DateType, BooleanType

In [0]:
# ============================================================================
# Step 1: Define Schema and Read Raw CSV Data
# ============================================================================
# Read retail_store_sales.csv from Unity Catalog volume
# Apply explicit schema to ensure data quality and type safety
# ============================================================================

schema = StructType([
    StructField("Transaction ID", StringType(), True),
    StructField("Customer ID", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Item", StringType(), True),
    StructField("Price Per Unit", DoubleType(), True),
    StructField("Quantity", DoubleType(), True),  # Keep as Double to preserve source precision
    StructField("Total Spent", DoubleType(), True),
    StructField("Payment Method", StringType(), True),
    StructField("Location", StringType(), True),
    StructField("Transaction Date", DateType(), True),
    StructField("Discount Applied", BooleanType(), True)
])

sales_df = spark.read \
                .option("header", True) \
                .schema(schema) \
                .csv(f"/Volumes/sales/default/raw/{table}.csv")

display(sales_df)

In [0]:
# ============================================================================
# Step 2: Standardize Column Names
# ============================================================================
# Convert column names from "Title Case" to "snake_case" convention
# ============================================================================

sales_df = sales_df.withColumnsRenamed({
    "Transaction ID": "transaction_id",
    "Customer ID": "customer_id",
    "Category": "category",
    "Item": "item_id",
    "Price Per Unit": "price_per_unit",
    "Quantity": "quantity",
    "Total Spent": "total_spent",
    "Payment Method": "payment_method",
    "Location": "location",
    "Transaction Date": "transaction_date",
    "Discount Applied": "discount_applied"
})

display(sales_df)

In [0]:
# ============================================================================
# Step 3: Write to Bronze Layer
# ============================================================================
# Save the raw data with standardized column names to the bronze layer
# Target: sales.bronze.retail_store_sales
# ============================================================================

sales_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"sales.bronze.{table}")